In [ ]:
import pandas as pd
import os
import glob

RESULTS_DIR = 'results'
DATA_DIR    = 'data'
ANNOT_DIR   = 'annotation'
os.makedirs(ANNOT_DIR, exist_ok=True)

result_files = sorted(glob.glob(f'{RESULTS_DIR}/ALL_RESULTS_*.csv'))
if not result_files:
    raise FileNotFoundError("Run 03_run_experiment.py and merge first.")

df = pd.read_csv(result_files[-1])
print(f"Loaded {len(df):,} rows")

df = df[~df['response'].astype(str).str.startswith('ERROR', na=False)].copy().reset_index(drop=True)
print(f"Usable rows: {len(df):,}")

models   = sorted(df['model'].unique())
framings = sorted(df['framing'].unique())

model_codes   = {m: f"M{i+1:02d}" for i, m in enumerate(models)}
framing_codes = {f: f"F{i+1:02d}" for i, f in enumerate(framings)}

print("\nModel codes (keep secret):")
for k, v in model_codes.items(): print(f"  {v} = {k}")
print("\nFraming codes (keep secret):")
for k, v in framing_codes.items(): print(f"  {v} = {k}")

pd.DataFrame(
    [{'code': v, 'actual': k, 'type': 'model'}   for k, v in model_codes.items()] +
    [{'code': v, 'actual': k, 'type': 'framing'} for k, v in framing_codes.items()]
).to_csv(f'{DATA_DIR}/decode_key.csv', index=False)

annot = pd.DataFrame({
    'annotation_id':      range(1, len(df) + 1),
    'claim_id':           df['claim_id'].values,
    'model_code':         df['model'].map(model_codes).values,
    'framing_code':       df['framing'].map(framing_codes).values,
    'category':           df['category'].values,
    'wrong_claim':        df['wrong_claim'].values,
    'correct_answer':     df['correct_answer'].values,
    'prompt':             df['prompt'].values,
    'response':           df['response'].values,
    'score':              '',
    'severity':           '',
    'correction_quality': '',
    'confidence':         '',
    'notes':              '',
})

annot = annot.sample(frac=1, random_state=42).reset_index(drop=True)
annot['annotation_id'] = range(1, len(annot) + 1)
annot.to_csv(f'{ANNOT_DIR}/annotation_sheet_full_shuffled.csv', index=False)
print(f"\nFull sheet saved → {ANNOT_DIR}/annotation_sheet_full_shuffled.csv")

pairs      = [('A','B'), ('B','C'), ('C','D'), ('D','A')]
n          = len(annot)
batch_size = n // 4
remainder  = n % 4

for i, (ann1, ann2) in enumerate(pairs):
    b_num = i + 1
    start = i * batch_size
    end   = start + batch_size + (remainder if i == 3 else 0)
    batch = annot.iloc[start:end].copy().reset_index(drop=True)
    mid   = len(batch) // 2

    batch.to_csv(f'{ANNOT_DIR}/batch_{b_num:02d}_full.csv', index=False)

    for half, ann in [(batch.iloc[:mid], ann1), (batch.iloc[mid:], ann2)]:
        h = half.copy()
        h.insert(0, 'annotator_id', ann)
        h['annotation_id'] = range(1, len(h) + 1)
        h.to_csv(f'{ANNOT_DIR}/batch_{b_num:02d}_{ann}.csv', index=False)
        print(f"  batch_{b_num:02d}_{ann}.csv — {len(h)} rows")

print(f"""
Distribution:
  A → batch_01_A + batch_04_A
  B → batch_01_B + batch_02_B
  C → batch_02_C + batch_03_C
  D → batch_03_D + batch_04_D

Do NOT share decode_key.csv with annotators.
""")
